# Salidas tabulares de consultas principales

Este cuaderno ejecuta 15 consultas analíticas y muestra sus resultados en tablas (DataFrames) para facilitar validación y revisión de datos.

In [10]:
from pathlib import Path
import subprocess
import tempfile

import pandas as pd
from IPython.display import display

DB_PATH = "localhost:/Users/wilsonjonatan/Documents/bases 1 2026/violencia guate/sql/db/violencia_guate.fdb"
DB_USER = "sysdba"
DB_PASSWORD = "masterkey"
ISQL_PATH = "/Library/Frameworks/Firebird.framework/Resources/bin/isql"

In [11]:
def run_query(sql: str) -> pd.DataFrame:
    query = sql.strip().rstrip(";") + ";"
    script = f"SET HEADING OFF;\nSET LIST ON;\n{query}\n"

    with tempfile.NamedTemporaryFile("w", suffix=".sql", delete=False, encoding="utf-8") as tmp:
        tmp.write(script)
        tmp_path = tmp.name

    try:
        result = subprocess.run(
            [ISQL_PATH, "-user", DB_USER, "-password", DB_PASSWORD, DB_PATH, "-q", "-i", tmp_path],
            capture_output=True,
            text=True,
        )
    finally:
        Path(tmp_path).unlink(missing_ok=True)

    output = "\n".join(part for part in (result.stdout, result.stderr) if part)
    rows = []
    current = {}

    for raw_line in output.splitlines():
        line = raw_line.strip()
        if not line:
            if current:
                rows.append(current)
                current = {}
            continue
        if line.startswith("SQL>") or line.startswith("DatabaseError") or line.startswith("Warning"):
            continue

        parts = line.split(None, 1)
        if len(parts) == 2:
            key, value = parts
            current[key.lower()] = value.strip()

    if current:
        rows.append(current)

    df = pd.DataFrame(rows)
    for column in df.columns:
        numeric = pd.to_numeric(df[column], errors="coerce")
        if len(df[column]) > 0 and numeric.notna().all():
            df[column] = numeric
    return df


def show_table(df: pd.DataFrame, titulo: str, max_rows: int = 200) -> None:
    print(f"\n{titulo}")
    print("-" * len(titulo))
    if df.empty:
        print("Sin datos para mostrar.")
        return
    print(f"Filas totales: {len(df)}")
    display(df.head(max_rows))
    if len(df) > max_rows:
        print(f"Mostrando primeras {max_rows} filas de {len(df)}.")

In [12]:
# Ejecutar las 30 consultas propuestas en el mismo orden del archivo SQL
import re

sql_file = Path("/Users/wilsonjonatan/Documents/bases 1 2026/violencia guate/sql/consultas_propuestas.sql")
contenido = sql_file.read_text(encoding="utf-8")

patron = re.compile(
    r"/\*\s*(\d+)\)\s*(.*?)\*/\s*(.*?);(?=\s*/\*|\s*$)",
    re.S,
 )

consultas = []
for numero, titulo, sql in patron.findall(contenido):
    consultas.append((int(numero), titulo.strip(), sql.strip()))

consultas = sorted(consultas, key=lambda x: x[0])

print(f"Consultas detectadas: {len(consultas)}")

for numero, titulo, sql in consultas:
    df = run_query(sql)
    show_table(df, f"{numero}) {titulo}")

Consultas detectadas: 30

1) LISTA: Cantidad de homicidios por anio y departamento
--------------------------------------------------------
Filas totales: 289


,anio,departamento,total_homicidios
0,2023,Guatemala,20352
1,2023,Jutiapa,9042
2,2023,Quetzaltenango,6038
3,2023,Escuintla,4405
4,2023,Jalapa,3334
...,...,...,...
195,2016,Retalhuleu,4
196,2016,Huehuetenango,4
197,2016,Quetzaltenango,4
198,2014,San Marcos,4


Mostrando primeras 200 filas de 289.

2) LISTA/PARCIAL: Denuncias por violencia contra la mujer por municipio
-----------------------------------------------------------------------
Filas totales: 15


,departamento,municipio,total_denuncias,clasificacion_delito,delito
0,Alta Verapaz,Cobán,888,Violencia contra la mujer,Violencia contra la mujer
1,Alta Verapaz,San Pedro Carchá,250,Violencia contra la mujer,Violencia contra la mujer
2,Alta Verapaz,San Cristóbal Verapaz,180,Violencia contra la mujer,Violencia contra la mujer
3,Alta Verapaz,Fray Bartolomé de Las Casas,100,Violencia contra la mujer,Violencia contra la mujer
4,Alta Verapaz,Raxruhá,81,Violencia contra la mujer,Violencia contra la mujer
5,Alta Verapaz,Chahal,76,Violencia contra la mujer,Violencia contra la mujer
6,Alta Verapaz,Chisec,74,Violencia contra la mujer,Violencia contra la mujer
7,Alta Verapaz,Santa Cruz Verapaz,71,Violencia contra la mujer,Violencia contra la mujer
8,Alta Verapaz,Tactic,57,Violencia contra la mujer,Violencia contra la mujer
9,Alta Verapaz,San Juan Chamelco,39,Violencia contra la mujer,Violencia contra la mujer



3) LISTA: Top 5 tipos de hechos delictivos en ultimos 5 anios
-------------------------------------------------------------
Filas totales: 5


...


25) PARCIAL: Top 5 municipios Chagas+Zika+Chikungunya combinados
----------------------------------------------------------------
Sin datos para mostrar.

26) PARCIAL: Vectores vs urbanizacion por departamento
------------------------------------------------------
Sin datos para mostrar.

27) PARCIAL: Tasa de morbilidad materna infantil por anio/departamento
----------------------------------------------------------------------
Sin datos para mostrar.

28) PARCIAL: Correlacion desnutricion y hechos delictivos por municipio
   Esta salida deja la serie lista para calcular correlacion fuera de SQL.
--------------------------------------------------------------------------------------------------------------------------------------------------
Filas totales: 1676


,anio,departamento,municipio,casos_desnutricion,hechos_delictivos
0,1962,Guatemala,Mixco,0,1
1,1967,Chiquimula,Chiquimula,0,2
2,1972,Guatemala,Guatemala,0,1
3,1980,Jutiapa,Yupiltepeque,0,2
4,1984,Quiché,Chajul,0,3
...,...,...,...,...,...
195,2010,Jalapa,Jalapa,0,84
196,2010,Jalapa,San Luis Jilotepeque,0,2
197,2010,Jutiapa,Jutiapa,0,37
198,2010,Jutiapa,Quesada,0,6


Mostrando primeras 200 filas de 1676.

29) LISTA/PARCIAL: Violencia estructural urbano vs rural
--------------------------------------------------------
Filas totales: 16


,anio,area_geografica,total_casos
0,2016,Urbano,22
1,2016,SIN_AREA,3
2,2017,Urbano,22
3,2017,SIN_AREA,2
4,2018,Urbano,18
5,2018,Rural,2
6,2018,SIN_AREA,2
7,2019,Urbano,18
8,2019,Rural,1
9,2019,SIN_AREA,1



30) LISTA/PARCIAL: Frecuencia de exhumaciones por tipo de delito relacionado
----------------------------------------------------------------------------
Sin datos para mostrar.
